<a href="https://colab.research.google.com/github/dee431/-Multimodal-Video-Aware-Search-Engine-with-RAG-/blob/main/Multimodal_Video_Aware_Search_Engine_with_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Multimodal Video-Aware Search Engine with RAG (Colab Demo)
# ============================================================

# ---------------------------
# 1. Install dependencies
# ---------------------------
!pip install -q opencv-python yt-dlp faster-whisper sentence-transformers faiss-cpu transformers accelerate

# ---------------------------
# 2. Imports
# ---------------------------
import os
import subprocess
import numpy as np
import cv2
from datetime import timedelta
from faster_whisper import WhisperModel
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from PIL import Image

# ---------------------------
# 3. Download and Re-encode video for OpenCV compatibility
# ---------------------------
video_url = "https://www.youtube.com/watch?v=MMmOLN5zBLY"
temp_video = os.path.abspath('temp_video.mp4')
video_path = os.path.abspath('sample_video.mp4')

print("Downloading video using yt-dlp...")
subprocess.run([
    "yt-dlp",
    "-f", "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best",
    "--merge-output-format", "mp4",
    "-o", temp_video,
    video_url
], capture_output=True)

# Re-encode to ensure compatibility with OpenCV's FFMPEG backend
print("Re-encoding video for compatibility...")
subprocess.run([
    "ffmpeg", "-i", temp_video, "-vcodec", "libx264", "-acodec", "aac", "-y", video_path
], capture_output=True)

if not os.path.exists(video_path):
    raise FileNotFoundError(f"Failed to process video to {video_path}")
print(f"Video ready at: {video_path}")

# ---------------------------
# 4. Extract audio and transcribe with Whisper
# ---------------------------
print("Loading Whisper (small model)...")
device = "cuda" if os.environ.get("CUDA_VISIBLE_DEVICES") else "cpu"
compute_type = "float32" if device == "cpu" else "float16"
whisper_model = WhisperModel("small", device=device, compute_type=compute_type)

audio_path = os.path.abspath("audio.mp3")
subprocess.run(["ffmpeg", "-i", video_path, "-vn", "-acodec", "libmp3lame", "-q:a", "2", audio_path, "-y"], capture_output=True)

print("Transcribing audio...")
segments, info = whisper_model.transcribe(audio_path, word_timestamps=True)
transcription_segments = list(segments)
print(f"Transcription done. Detected language: {info.language}, duration: {info.duration} seconds")

# ---------------------------
# 5. Extract frames at regular intervals
# ---------------------------
print("Extracting frames...")
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise IOError(f"Could not open video file: {video_path}")

video_duration = info.duration
frame_interval = 2.0
frames = []
timestamps = []
current_time = 0.0

while current_time < video_duration:
    cap.set(cv2.CAP_PROP_POS_MSEC, current_time * 1000)
    ret, frame = cap.read()
    if ret:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame_rgb)
        timestamps.append(current_time)
    current_time += frame_interval
cap.release()

if not frames:
    raise ValueError("No frames were extracted. Re-encoding failed to resolve decoding issues.")
print(f"Extracted {len(frames)} frames.")

# ---------------------------
# 6. Load CLIP model for multimodal embeddings
# ---------------------------
print("Loading CLIP (ViT-B/32)...")
clip_model = SentenceTransformer('clip-ViT-B-32')

# ---------------------------
# 7. Create multimodal chunks
# ---------------------------
chunks = []
for seg in transcription_segments:
    text = seg.text.strip()
    if not text:
        continue

    mid = (seg.start + seg.end) / 2.0
    closest_idx = min(range(len(timestamps)), key=lambda i: abs(timestamps[i] - mid))
    frame = frames[closest_idx]

    image_embedding = clip_model.encode(Image.fromarray(frame))
    text_embedding = clip_model.encode(text)

    chunks.append({
        "start": seg.start,
        "end": seg.end,
        "text": text,
        "image_embedding": image_embedding,
        "text_embedding": text_embedding
    })
print(f"Created {len(chunks)} multimodal chunks.")

# ---------------------------
# 8. Build FAISS index
# ---------------------------
dim = clip_model.get_sentence_embedding_dimension()
index = faiss.IndexFlatIP(dim)
text_embs = np.array([c["text_embedding"] for c in chunks], dtype=np.float32)
faiss.normalize_L2(text_embs)
index.add(text_embs)

# ---------------------------
# 9. Define search function
# ---------------------------
def search(query, top_k=3, alpha=0.7):
    query_emb = clip_model.encode(query, convert_to_tensor=False).astype(np.float32)
    faiss.normalize_L2(query_emb.reshape(1, -1))

    D, I = index.search(query_emb.reshape(1, -1), top_k * 2)
    candidates = [chunks[i] for i in I[0]]

    scores = []
    for chunk in candidates:
        t_sim = np.dot(query_emb, chunk["text_embedding"] / np.linalg.norm(chunk["text_embedding"]))
        i_norm = chunk["image_embedding"] / np.linalg.norm(chunk["image_embedding"])
        i_sim = np.dot(query_emb, i_norm)
        scores.append((alpha * t_sim + (1 - alpha) * i_sim, chunk))

    scores.sort(key=lambda x: x[0], reverse=True)
    return [s[1] for s in scores[:top_k]]

# ---------------------------
# 10. LLM Answer Generation
# ---------------------------
print("Loading Flan-T5-base...")
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def multimodal_video_qa(query):
    print(f"\n🔍 Query: '{query}'")
    res_chunks = search(query)
    context = " ".join([f"[{timedelta(seconds=int(c['start']))}] {c['text']}" for c in res_chunks])
    inputs = tokenizer(f"question: {query} context: {context}", return_tensors="pt", truncation=True, max_length=512)
    outputs = llm.generate(**inputs, max_new_tokens=80)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"🤖 Answer: {answer}\n")

# ---------------------------
# 11. Test
# ---------------------------
queries = ["How does a bilingual brain differ from a monolingual one?", "What are the benefits of speaking multiple languages?"]
for q in queries:
    multimodal_video_qa(q)
print("🎉 All done!")

Re-encoding video for compatibility...
Video ready at: /content/sample_video.mp4
Loading Whisper (small model)...
Transcribing audio...
Transcription done. Detected language: en, duration: 303.7866875 seconds
Extracting frames...
Extracted 152 frames.
Loading CLIP (ViT-B/32)...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Created 64 multimodal chunks.
Loading Flan-T5-base...


/tmp/ipykernel_681/613553166.py:128: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = clip_model.get_sentence_embedding_dimension()


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


🔍 Query: 'How does a bilingual brain differ from a monolingual one?'
🤖 Answer: more


🔍 Query: 'What are the benefits of speaking multiple languages?'
🤖 Answer: [0:03:04]

🎉 All done!
